# ApprovalMax v2 CDC Automation

Serverless-compatible notebook task. No classic cluster config and no low-level SparkContext APIs.

In [ ]:
from datetime import datetime, timezone
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
from pyspark.sql.window import Window

catalog = 'approvalmax_ai_platform'
schemas = {
    'bronze': 'bronze',
    'silver': 'silver',
    'vault': 'vault',
    'gold': 'gold',
    'quarantine': 'quarantine',
    'monitoring': 'monitoring',
}

spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}')
for schema_name in schemas.values():
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}')

def tbl(layer, name):
    return f'{catalog}.{schemas[layer]}.{name}'

run_id = f'cdc_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}'
audit_table = tbl('monitoring', 'etl_audit_log')
audit_schema = StructType([
    StructField('run_id', StringType(), False),
    StructField('step_name', StringType(), False),
    StructField('source_layer', StringType(), True),
    StructField('target_layer', StringType(), True),
    StructField('source_table', StringType(), True),
    StructField('target_table', StringType(), True),
    StructField('status', StringType(), False),
    StructField('source_row_count', LongType(), True),
    StructField('target_row_count', LongType(), True),
    StructField('failed_row_count', LongType(), True),
    StructField('started_at', TimestampType(), False),
    StructField('ended_at', TimestampType(), False),
    StructField('error_message', StringType(), True),
])

spark.sql(f'''CREATE TABLE IF NOT EXISTS {audit_table} (
  run_id STRING, step_name STRING, source_layer STRING, target_layer STRING,
  source_table STRING, target_table STRING, status STRING,
  source_row_count BIGINT, target_row_count BIGINT, failed_row_count BIGINT,
  started_at TIMESTAMP, ended_at TIMESTAMP, error_message STRING
) USING DELTA''')

def count_table(table_name):
    try:
        return spark.table(table_name).count()
    except Exception:
        return None

def write_audit(step_name, source_layer, target_layer, source_table, target_table, status, source_count, target_count, failed_count=0, error=None, started_at=None):
    now = datetime.now(timezone.utc)
    record = [{
        'run_id': run_id,
        'step_name': step_name,
        'source_layer': source_layer,
        'target_layer': target_layer,
        'source_table': source_table,
        'target_table': target_table,
        'status': status,
        'source_row_count': source_count,
        'target_row_count': target_count,
        'failed_row_count': failed_count,
        'started_at': started_at or now,
        'ended_at': now,
        'error_message': error,
    }]
    spark.createDataFrame(record, schema=audit_schema).write.format('delta').mode('append').saveAsTable(audit_table)

def audited_step(step_name, source_layer, target_layer, source_table, target_table, fn, failed_count=0):
    started_at = datetime.now(timezone.utc)
    source_count = count_table(source_table) if source_table else None
    try:
        fn()
        target_count = count_table(target_table) if target_table else None
        write_audit(step_name, source_layer, target_layer, source_table, target_table, 'success', source_count, target_count, failed_count, started_at=started_at)
    except Exception as exc:
        write_audit(step_name, source_layer, target_layer, source_table, target_table, 'failed', source_count, None, failed_count, str(exc), started_at)
        raise

def latest_by_key(df, key_col):
    w = Window.partitionBy(key_col).orderBy(F.col('sequence_id').desc(), F.col('ingestion_timestamp').desc())
    return df.withColumn('_rn', F.row_number().over(w)).filter(F.col('_rn') == 1).drop('_rn')


In [ ]:
records = [
    {'schema':'approvalmax_cdc_v1','source_system':'approvalmax_mock_cdc','source_table':'companies','op':'c','sequence_id':100001,'event_timestamp':'2026-05-01T09:00:00Z','ingestion_timestamp':'2026-05-01T09:00:20Z','primary_key':{'company_id':'COMP-001'},'before':None,'after':{'company_id':'COMP-001','company_name':'Northwind Finance Ltd','country':'GB','base_currency':'GBP','status':'active','created_at':'2026-04-01T08:00:00Z','updated_at':'2026-05-01T09:00:00Z'},'metadata':{'connector':'mock_debezium_style','database':'approvalmax_operational','lsn':100001,'tx_id':'TX-010001','is_snapshot':False}},
    {'schema':'approvalmax_cdc_v1','source_system':'approvalmax_mock_cdc','source_table':'users','op':'c','sequence_id':100002,'event_timestamp':'2026-05-01T09:05:00Z','ingestion_timestamp':'2026-05-01T09:05:20Z','primary_key':{'user_id':'USR-001','company_id':'COMP-001'},'before':None,'after':{'user_id':'USR-001','company_id':'COMP-001','full_name':'Priya Shah','email':'priya@example.com','role':'approver','is_active':True},'metadata':{'connector':'mock_debezium_style','database':'approvalmax_operational','lsn':100002,'tx_id':'TX-010002','is_snapshot':False}},
    {'schema':'approvalmax_cdc_v1','source_system':'approvalmax_mock_cdc','source_table':'subscriptions','op':'c','sequence_id':100003,'event_timestamp':'2026-05-01T09:10:00Z','ingestion_timestamp':'2026-05-01T09:10:20Z','primary_key':{'subscription_id':'SUB-001','company_id':'COMP-001'},'before':None,'after':{'subscription_id':'SUB-001','company_id':'COMP-001','plan_name':'Enterprise','mrr':1200.00,'status':'active','started_at':'2026-04-01T00:00:00Z','cancelled_at':None},'metadata':{'connector':'mock_debezium_style','database':'approvalmax_operational','lsn':100003,'tx_id':'TX-010003','is_snapshot':False}},
    {'schema':'approvalmax_cdc_v1','source_system':'approvalmax_mock_cdc','source_table':'finance_documents','op':'u','sequence_id':100123,'event_timestamp':'2026-05-01T10:00:00Z','ingestion_timestamp':'2026-05-01T10:00:20Z','primary_key':{'document_id':'PO-001','company_id':'COMP-001'},'before':{},'after':{'document_id':'PO-001','company_id':'COMP-001','document_type':'purchase_order','document_status':'approved','supplier_id':'SUP-001','supplier_name':'Acme Supplies','total_amount':850.25,'currency':'GBP','created_at':'2026-05-01T09:20:00Z','submitted_at':'2026-05-01T09:30:00Z','approved_at':'2026-05-01T10:00:00Z','rejected_at':None,'paid_at':None,'updated_at':'2026-05-01T10:00:00Z','approval_deadline_at':'2026-05-01T12:00:00Z'},'metadata':{'connector':'mock_debezium_style','database':'approvalmax_operational','lsn':100123,'tx_id':'TX-010012','is_snapshot':False}},
    {'schema':'approvalmax_cdc_v1','source_system':'approvalmax_mock_cdc','source_table':'approval_events','op':'c','sequence_id':100124,'event_timestamp':'2026-05-01T09:30:00Z','ingestion_timestamp':'2026-05-01T09:30:20Z','primary_key':{'event_id':'EVT-001','document_id':'PO-001'},'before':None,'after':{'event_id':'EVT-001','document_id':'PO-001','workflow_id':'WF-001','company_id':'COMP-001','approver_user_id':'USR-001','event_type':'submitted','event_timestamp':'2026-05-01T09:30:00Z'},'metadata':{'connector':'mock_debezium_style','database':'approvalmax_operational','lsn':100124,'tx_id':'TX-010013','is_snapshot':False}},
    {'schema':'approvalmax_cdc_v1','source_system':'approvalmax_mock_cdc','source_table':'approval_events','op':'c','sequence_id':100125,'event_timestamp':'2026-05-01T10:00:00Z','ingestion_timestamp':'2026-05-01T10:00:20Z','primary_key':{'event_id':'EVT-002','document_id':'PO-001'},'before':None,'after':{'event_id':'EVT-002','document_id':'PO-001','workflow_id':'WF-001','company_id':'COMP-001','approver_user_id':'USR-001','event_type':'approved','event_timestamp':'2026-05-01T10:00:00Z'},'metadata':{'connector':'mock_debezium_style','database':'approvalmax_operational','lsn':100125,'tx_id':'TX-010014','is_snapshot':False}},
]

bronze_table = tbl('bronze', 'approvalmax_cdc_raw')
raw_json_schema = StructType([StructField('raw_json', StringType(), False)])
cdc_schema = '''
STRUCT<
  schema: STRING,
  source_system: STRING,
  source_table: STRING,
  op: STRING,
  sequence_id: BIGINT,
  event_timestamp: STRING,
  ingestion_timestamp: STRING,
  primary_key: MAP<STRING, STRING>,
  before: MAP<STRING, STRING>,
  after: STRUCT<
    company_id: STRING, company_name: STRING, country: STRING, base_currency: STRING, status: STRING,
    user_id: STRING, full_name: STRING, email: STRING, role: STRING, is_active: BOOLEAN,
    subscription_id: STRING, plan_name: STRING, mrr: DOUBLE, started_at: STRING, cancelled_at: STRING,
    document_id: STRING, document_type: STRING, document_status: STRING, supplier_id: STRING, supplier_name: STRING,
    total_amount: DOUBLE, currency: STRING, created_at: STRING, submitted_at: STRING, approved_at: STRING,
    rejected_at: STRING, paid_at: STRING, updated_at: STRING, approval_deadline_at: STRING,
    event_id: STRING, workflow_id: STRING, approver_user_id: STRING, event_type: STRING, event_timestamp: STRING
  >,
  metadata: STRUCT<connector: STRING, database: STRING, lsn: BIGINT, tx_id: STRING, is_snapshot: BOOLEAN>
>
'''

def write_bronze():
    raw_df = spark.createDataFrame(
        [{'raw_json': json.dumps(record)} for record in records],
        schema=raw_json_schema,
    )
    df = (
        raw_df
        .select('raw_json', F.from_json(F.col('raw_json'), cdc_schema).alias('record'))
        .select('record.*', F.col('raw_json').alias('_raw_json'))
        .withColumn('_loaded_at', F.current_timestamp())
        .withColumn('_run_id', F.lit(run_id))
    )
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(bronze_table)

audited_step('write_bronze_raw_cdc', 'source', 'bronze', 'inline_cdc_records', bronze_table, write_bronze)
bronze = spark.table(bronze_table)


In [ ]:
silver_specs = {
    'companies_current': latest_by_key(bronze.filter((F.col('source_table') == 'companies') & (F.col('op') != 'd')).select(F.col('after.company_id').alias('company_id'), F.col('after.company_name').alias('company_name'), F.col('after.country').alias('country'), F.col('after.base_currency').alias('base_currency'), F.col('after.status').alias('company_status'), F.col('after.created_at').cast('timestamp').alias('created_at'), F.col('after.updated_at').cast('timestamp').alias('updated_at'), F.col('sequence_id'), F.col('ingestion_timestamp').cast('timestamp').alias('ingestion_timestamp'), F.col('source_system').alias('record_source')), 'company_id'),
    'users_current': latest_by_key(bronze.filter((F.col('source_table') == 'users') & (F.col('op') != 'd')).select(F.col('after.user_id').alias('user_id'), F.col('after.company_id').alias('company_id'), F.col('after.full_name').alias('full_name'), F.col('after.email').alias('email'), F.col('after.role').alias('role'), F.col('after.is_active').cast('boolean').alias('is_active'), F.col('sequence_id'), F.col('ingestion_timestamp').cast('timestamp').alias('ingestion_timestamp'), F.col('source_system').alias('record_source')), 'user_id'),
    'subscriptions_current': latest_by_key(bronze.filter((F.col('source_table') == 'subscriptions') & (F.col('op') != 'd')).select(F.col('after.subscription_id').alias('subscription_id'), F.col('after.company_id').alias('company_id'), F.col('after.plan_name').alias('plan_name'), F.col('after.mrr').cast('decimal(18,2)').alias('mrr'), F.col('after.status').alias('subscription_status'), F.col('after.started_at').cast('timestamp').alias('started_at'), F.col('after.cancelled_at').cast('timestamp').alias('cancelled_at'), F.col('sequence_id'), F.col('ingestion_timestamp').cast('timestamp').alias('ingestion_timestamp'), F.col('source_system').alias('record_source')), 'subscription_id'),
    'finance_documents_current': latest_by_key(bronze.filter((F.col('source_table') == 'finance_documents') & (F.col('op') != 'd')).select(F.col('after.document_id').alias('document_id'), F.col('after.company_id').alias('company_id'), F.col('after.document_type').alias('document_type'), F.col('after.document_status').alias('document_status'), F.col('after.supplier_id').alias('supplier_id'), F.col('after.supplier_name').alias('supplier_name'), F.col('after.total_amount').cast('decimal(18,2)').alias('total_amount'), F.col('after.currency').alias('currency'), F.col('after.created_at').cast('timestamp').alias('created_at'), F.col('after.submitted_at').cast('timestamp').alias('submitted_at'), F.col('after.approved_at').cast('timestamp').alias('approved_at'), F.col('after.rejected_at').cast('timestamp').alias('rejected_at'), F.col('after.paid_at').cast('timestamp').alias('paid_at'), F.col('after.updated_at').cast('timestamp').alias('updated_at'), F.col('after.approval_deadline_at').cast('timestamp').alias('approval_deadline_at'), F.col('sequence_id'), F.col('ingestion_timestamp').cast('timestamp').alias('ingestion_timestamp'), F.col('source_system').alias('record_source')), 'document_id'),
    'approval_events': latest_by_key(bronze.filter((F.col('source_table') == 'approval_events') & (F.col('op') != 'd')).select(F.col('after.event_id').alias('event_id'), F.col('after.document_id').alias('document_id'), F.col('after.workflow_id').alias('workflow_id'), F.col('after.company_id').alias('company_id'), F.col('after.approver_user_id').alias('approver_user_id'), F.col('after.event_type').alias('event_type'), F.col('after.event_timestamp').cast('timestamp').alias('approval_event_timestamp'), F.col('sequence_id'), F.col('ingestion_timestamp').cast('timestamp').alias('ingestion_timestamp'), F.col('source_system').alias('record_source')), 'event_id'),
}

for name, df in silver_specs.items():
    target = tbl('silver', name)
    audited_step(f'build_silver_{name}', 'bronze', 'silver', bronze_table, target, lambda df=df, target=target: df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(target))


In [ ]:
statements = {
    tbl('vault','hub_company'): f"""CREATE OR REPLACE TABLE {tbl('vault','hub_company')} AS SELECT DISTINCT sha2(company_id, 256) AS company_hk, company_id AS company_bk, current_timestamp() AS load_datetime, record_source FROM {tbl('silver','companies_current')} WHERE company_id IS NOT NULL""",
    tbl('vault','hub_finance_document'): f"""CREATE OR REPLACE TABLE {tbl('vault','hub_finance_document')} AS SELECT DISTINCT sha2(document_id, 256) AS finance_document_hk, document_id AS finance_document_bk, current_timestamp() AS load_datetime, record_source FROM {tbl('silver','finance_documents_current')} WHERE document_id IS NOT NULL""",
    tbl('vault','hub_approval_event'): f"""CREATE OR REPLACE TABLE {tbl('vault','hub_approval_event')} AS SELECT DISTINCT sha2(event_id, 256) AS approval_event_hk, event_id AS approval_event_bk, current_timestamp() AS load_datetime, record_source FROM {tbl('silver','approval_events')} WHERE event_id IS NOT NULL""",
    tbl('vault','link_document_company'): f"""CREATE OR REPLACE TABLE {tbl('vault','link_document_company')} AS SELECT DISTINCT sha2(concat_ws('||', document_id, company_id), 256) AS document_company_hk, sha2(document_id, 256) AS finance_document_hk, sha2(company_id, 256) AS company_hk, current_timestamp() AS load_datetime, record_source FROM {tbl('silver','finance_documents_current')} WHERE document_id IS NOT NULL AND company_id IS NOT NULL""",
    tbl('vault','link_document_approval_event'): f"""CREATE OR REPLACE TABLE {tbl('vault','link_document_approval_event')} AS SELECT DISTINCT sha2(concat_ws('||', document_id, event_id), 256) AS document_approval_event_hk, sha2(document_id, 256) AS finance_document_hk, sha2(event_id, 256) AS approval_event_hk, current_timestamp() AS load_datetime, record_source FROM {tbl('silver','approval_events')} WHERE document_id IS NOT NULL AND event_id IS NOT NULL""",
    tbl('vault','sat_finance_document_status'): f"""CREATE OR REPLACE TABLE {tbl('vault','sat_finance_document_status')} AS SELECT sha2(document_id, 256) AS finance_document_hk, document_status, document_type, supplier_id, supplier_name, created_at, submitted_at, approved_at, rejected_at, paid_at, updated_at, approval_deadline_at, sha2(concat_ws('||', coalesce(document_status,''), coalesce(document_type,''), coalesce(cast(updated_at as string),'')), 256) AS hashdiff, current_timestamp() AS load_datetime, record_source FROM {tbl('silver','finance_documents_current')} WHERE document_id IS NOT NULL""",
    tbl('vault','sat_finance_document_amount'): f"""CREATE OR REPLACE TABLE {tbl('vault','sat_finance_document_amount')} AS SELECT sha2(document_id, 256) AS finance_document_hk, total_amount, currency, sha2(concat_ws('||', coalesce(cast(total_amount as string),''), coalesce(currency,'')), 256) AS hashdiff, current_timestamp() AS load_datetime, record_source FROM {tbl('silver','finance_documents_current')} WHERE document_id IS NOT NULL""",
    tbl('vault','sat_approval_event_detail'): f"""CREATE OR REPLACE TABLE {tbl('vault','sat_approval_event_detail')} AS SELECT sha2(event_id, 256) AS approval_event_hk, workflow_id, document_id, company_id, approver_user_id, event_type, approval_event_timestamp, sha2(concat_ws('||', coalesce(workflow_id,''), coalesce(document_id,''), coalesce(event_type,''), coalesce(cast(approval_event_timestamp as string),'')), 256) AS hashdiff, current_timestamp() AS load_datetime, record_source FROM {tbl('silver','approval_events')} WHERE event_id IS NOT NULL""",
}

for target, statement in statements.items():
    audited_step(f'build_vault_{target.split(".")[-1]}', 'silver', 'vault', None, target, lambda statement=statement: spark.sql(statement))


In [ ]:
gold_sql = [
    f"CREATE OR REPLACE TABLE {tbl('gold','dim_company')} AS SELECT company_id, company_name, country, base_currency, company_status FROM {tbl('silver','companies_current')}",
    f"CREATE OR REPLACE TABLE {tbl('gold','dim_user')} AS SELECT user_id, company_id, full_name, email, role, is_active FROM {tbl('silver','users_current')}",
    f"CREATE OR REPLACE TABLE {tbl('gold','dim_subscription')} AS SELECT subscription_id, company_id, plan_name, mrr, subscription_status, started_at, cancelled_at FROM {tbl('silver','subscriptions_current')}",
    f"CREATE OR REPLACE TABLE {tbl('gold','dim_finance_document')} AS SELECT document_id, company_id, document_type, document_status, supplier_id, supplier_name, currency FROM {tbl('silver','finance_documents_current')}",
    f"CREATE OR REPLACE TABLE {tbl('gold','dim_workflow')} AS SELECT DISTINCT workflow_id, company_id FROM {tbl('silver','approval_events')} WHERE workflow_id IS NOT NULL",
    f"""CREATE OR REPLACE TABLE {tbl('gold','fact_approval_document_lifecycle')} AS SELECT d.document_id, d.company_id, c.company_name, s.plan_name, d.document_type, d.document_status, d.supplier_id, d.supplier_name, d.total_amount, d.currency, d.created_at, d.submitted_at, d.approved_at, d.rejected_at, d.paid_at, d.updated_at, d.approval_deadline_at, CASE WHEN d.submitted_at IS NOT NULL AND d.approved_at IS NOT NULL THEN timestampdiff(MINUTE, d.submitted_at, d.approved_at) END AS approval_cycle_time_minutes, CASE WHEN d.approval_deadline_at IS NOT NULL AND d.approved_at IS NOT NULL AND d.approved_at > d.approval_deadline_at THEN 1 ELSE 0 END AS approval_sla_breach_flag FROM {tbl('silver','finance_documents_current')} d LEFT JOIN {tbl('silver','companies_current')} c ON d.company_id = c.company_id LEFT JOIN {tbl('silver','subscriptions_current')} s ON d.company_id = s.company_id""",
    f"CREATE OR REPLACE TABLE {tbl('gold','fact_approval_events')} AS SELECT event_id, document_id, workflow_id, company_id, approver_user_id, event_type, approval_event_timestamp FROM {tbl('silver','approval_events')}",
    f"CREATE OR REPLACE TABLE {tbl('gold','fact_cdc_context_summary')} AS SELECT source_table, op, count(*) AS event_count, min(cast(event_timestamp as timestamp)) AS first_event_at, max(cast(event_timestamp as timestamp)) AS last_event_at FROM {bronze_table} GROUP BY source_table, op",
]

for statement in gold_sql:
    target = statement.split(' TABLE ')[1].split(' AS ')[0]
    audited_step(f'build_gold_{target.split(".")[-1]}', 'silver', 'gold', None, target, lambda statement=statement: spark.sql(statement))

quality_checks = {
    'gold_document_id_not_null': f"SELECT * FROM {tbl('gold','fact_approval_document_lifecycle')} WHERE document_id IS NULL",
    'gold_company_id_not_null': f"SELECT * FROM {tbl('gold','fact_approval_document_lifecycle')} WHERE company_id IS NULL",
    'gold_document_status_valid': f"SELECT * FROM {tbl('gold','fact_approval_document_lifecycle')} WHERE document_status IS NULL OR document_status NOT IN ('draft','submitted','approved','rejected','paid','cancelled')",
    'gold_total_amount_non_negative': f"SELECT * FROM {tbl('gold','fact_approval_document_lifecycle')} WHERE total_amount < 0",
    'gold_cycle_time_non_negative': f"SELECT * FROM {tbl('gold','fact_approval_document_lifecycle')} WHERE approval_cycle_time_minutes < 0",
}

failures = []
for name, query in quality_checks.items():
    failed = spark.sql(query)
    failed_count = failed.count()
    if failed_count:
        qtable = tbl('quarantine', name)
        failed.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(qtable)
        failures.append(f'{name}: {failed_count}')
    write_audit(f'quality_check_{name}', 'gold', 'quarantine', 'sql_expectation', name, 'failed' if failed_count else 'success', None, failed_count, failed_count)

if failures:
    raise Exception('CDC automation quality checks failed: ' + '; '.join(failures))

print(f'CDC automation complete. run_id={run_id}')
